In [ ]:
import pandas as pd
import io

In [1]:
import pandas as pd
import os

# =============================
# 1. 参数设置 (已更新为您的新路径)
# =============================
INPUT_FILE = "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.4.cNMF_Mono_Macro/Example_refbuilder_Macro/ALL_cGEP_top_genes_Mono_Macro_wide_filt.csv"
OUT_DIR = "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.4.cNMF_Mono_Macro/3.1.Macro_GEP_Anno/"
OUT_FILENAME = "3.Mono_Macro_cGEP_Annotation_Results.csv"

# 确保输出目录存在
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
OUT_PATH = os.path.join(OUT_DIR, OUT_FILENAME)

# =============================
# 2. 定义参考集 (单核/巨噬细胞专用字典)
# =============================
marker_dict = {
    # --- 单核细胞亚群 (Monocyte Subtypes) ---
    'Mono_CD14_Classical': {'CD14', 'S100A8', 'S100A9', 'FCN1', 'VCAN', 'CCR2'},
    'Mono_CD16_NonClassical': {'FCGR3A', 'LST1', 'CX3CR1', 'LILRA5', 'MS4A7'},

    # --- 巨噬细胞核心亚群/状态 (Macrophage Functional States) ---
    # 促血管生成/基质重塑 (M2-like bad, 缺氧相关)
    'Macro_SPP1_Angio': {'SPP1', 'FN1', 'VEGFA', 'PLAUR', 'MMP9', 'MMP14', 'TGFB1'},
    # T细胞招募/抗肿瘤 (M1-like good, IFN-g 驱动)
    'Macro_CXCL9_Immune': {'CXCL9', 'CXCL10', 'CXCL11', 'GBP1', 'STAT1', 'CD40', 'CD274'},
    # 抗原提呈/组织驻留 (Baseline, MHC-II & C1Q 高表达)
    'Macro_C1QC_Resident': {'C1QA', 'C1QB', 'C1QC', 'HLA-DRA', 'CD74', 'APOE', 'CD163'},
    # 脂质代谢/泡沫样 (LAMs, 代谢活跃)
    'Macro_TREM2_Lipid': {'TREM2', 'LIPA', 'GPNMB', 'APOE', 'FABP4', 'LGALS3'},
    
    # --- 特殊功能模块 (Special Modules) ---
    # 干扰素响应 (ISG+, Type I IFN 驱动)
    'Macro_IFIT1_Interferon': {'IFIT1', 'ISG15', 'MX1', 'IFI44L', 'RSAD2', 'OAS1'},
    # 促炎细胞因子 (NFkB-driven, 快速炎症)
    'Macro_IL1B_Inflam': {'IL1B', 'CXCL8', 'CCL3', 'CCL4', 'NFKBIA', 'EREG'},
    # 细胞增殖 (Cycling)
    'Macro_Cycling': {'MKI67', 'TOP2A', 'PCNA', 'CDK1', 'STMN1'},
    # 金属硫蛋白/应激 (Stress)
    'Macro_Metallothionein': {'MT1G', 'MT1X', 'MT2A', 'MT1H'},

    # --- 质控与双胞剔除 (QC & Doublets) ---
    # 必须剔除的异源污染
    'Doublet_T_Cell': {'CD3D', 'CD3E', 'CD247', 'TRBC1', 'TRBC2', 'LCK'},
    'Doublet_B_Cell': {'MS4A1', 'CD79A', 'CD79B'},  # 不含 CD74 (它是巨噬 Marker)
    'Doublet_NK_Cell': {'NKG7', 'GNLY', 'PRF1', 'KLRD1'},
    'Doublet_Epithelial': {'EPCAM', 'KRT8', 'KRT18', 'KRT19', 'KRT14'},
    'Doublet_Fibroblast': {'COL1A1', 'COL1A2', 'DCN', 'LUM'},
    'Doublet_Endothelial': {'PECAM1', 'VWF', 'KDR', 'ENG'},
    
    # 环境噪音/非特异性信号
    'Artifact_RBC': {'HBB', 'HBA1', 'HBA2'}
}

print(f"✅ 成功加载 {len(marker_dict)} 个 Macrophage/Monocyte 参考类型。")

# =============================
# 3. 读取数据
# =============================
print(f"正在读取文件: {INPUT_FILE}")

try:
    df = pd.read_csv(INPUT_FILE)
    print(f"文件读取成功，包含 {len(df.columns)} 个 cGEP (列)。")
except FileNotFoundError:
    print(f"❌ 错误：找不到文件 {INPUT_FILE}")
    print("⚠️ 请检查路径是否存在。")
    exit()

# =============================
# 4. 注释函数
# =============================
def annotate_cgep_spectra(gene_list):
    input_genes = set(gene_list)
    
    best_results = {
        "Spectra_Label": "Unknown",
        "Overlap_Count": 0,
        "Jaccard_Score": 0.0,
        "Key_Markers": ""
    }
    
    max_score = -1
    
    for ref_name, ref_genes in marker_dict.items():
        # 计算交集
        intersection = input_genes & ref_genes
        if not intersection:
            continue
            
        overlap_n = len(intersection)
        
        # --- 评分算法 (Jaccard) ---
        jaccard = overlap_n / len(input_genes | ref_genes)
        score = jaccard
        
        if score > max_score:
            max_score = score
            best_results = {
                "Spectra_Label": ref_name,
                "Overlap_Count": overlap_n,
                "Jaccard_Score": round(score, 4),
                "Key_Markers": ",".join(list(intersection)) 
            }
            
    return best_results

# =============================
# 5. 执行注释循环
# =============================
results = []

# 遍历每一列 (因为文件名带 _wide_，假设列名是 cGEP 名称)
for cgep_col in df.columns:
    # 1. 获取当前 cGEP 的基因，去除空值，转为字符串并大写
    genes = (
        df[cgep_col]
        .dropna()
        .astype(str)
        .str.upper() 
        .tolist()
    )
    
    # 2. 如果该列没有基因，跳过
    if not genes:
        continue

    # 3. 运行注释
    anno = annotate_cgep_spectra(genes)
    
    # 4. 收集结果
    results.append({
        "cGEP_Name": cgep_col,
        "Spectra_Label": anno["Spectra_Label"],
        "Overlap_Count": anno["Overlap_Count"],
        "Jaccard_Score": anno["Jaccard_Score"],
        "Key_Markers": anno["Key_Markers"],
        "Top_5_Genes_Input": ",".join(genes[:5]) # 方便核对
    })

# =============================
# 6. 输出结果
# =============================
out_df = pd.DataFrame(results)

# 按照分数降序排列
if not out_df.empty:
    out_df = out_df.sort_values(by="Jaccard_Score", ascending=False)
    
    # 保存
    out_df.to_csv(OUT_PATH, index=False)

    print("-" * 30)
    print("✅ 单核/巨噬细胞 GEP 注释完成！")
    print(f"📄 结果已保存至: {OUT_PATH}")
    print("-" * 30)
    print("预览前 5 行结果:")
    print(out_df[["cGEP_Name", "Spectra_Label", "Overlap_Count", "Jaccard_Score"]].head())
else:
    print("⚠️ 警告：没有生成任何注释结果，请检查输入文件格式或基因名是否匹配。")

✅ 成功加载 17 个 Macrophage/Monocyte 参考类型。
正在读取文件: /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.4.cNMF_Mono_Macro/Example_refbuilder_Macro/ALL_cGEP_top_genes_Mono_Macro_wide_filt.csv
文件读取成功，包含 77 个 cGEP (列)。
------------------------------
✅ 单核/巨噬细胞 GEP 注释完成！
📄 结果已保存至: /sibcb1/bioinformatics/yangyue/project/immunotherapy/7.4.cNMF_Mono_Macro/3.1.Macro_GEP_Anno/3.Mono_Macro_cGEP_Annotation_Results.csv
------------------------------
预览前 5 行结果:
   cGEP_Name           Spectra_Label  Overlap_Count  Jaccard_Score
25    cGEP26  Macro_IFIT1_Interferon              6         0.2000
10    cGEP11          Doublet_T_Cell              6         0.2000
66    cGEP67       Macro_TREM2_Lipid              5         0.1613
48    cGEP49       Macro_IL1B_Inflam              5         0.1613
15    cGEP16       Macro_TREM2_Lipid              5         0.1613


In [2]:
out_df

,cGEP_Name,Spectra_Label,Overlap_Count,Jaccard_Score,Key_Markers,Top_5_Genes_Input
25,cGEP26,Macro_IFIT1_Interferon,6,0.2000,"MX1,IFI44L,IFIT1,RSAD2,OAS1,ISG15","ISG15,GBP1,TNFSF10,IFIT3,CXCL10"
10,cGEP11,Doublet_T_Cell,6,0.2000,"CD3D,TRBC1,CD3E,CD247,TRBC2,LCK","AC245427.1,IL32,CD3D,CD3E,PTPRCAP"
66,cGEP67,Macro_TREM2_Lipid,5,0.1613,"LIPA,APOE,FABP4,TREM2,GPNMB","RP11-160O5.1,FABP4,GPC4,SPOCD1,CRABP2"
48,cGEP49,Macro_IL1B_Inflam,5,0.1613,"CCL3,NFKBIA,CCL4,IL1B,CXCL8","CCL3L3,CCL3,CCL4,IL1B,NFKBIA"
15,cGEP16,Macro_TREM2_Lipid,5,0.1613,"LIPA,APOE,LGALS3,TREM2,GPNMB","APOE,CTSD,GPNMB,APOC1,RP11-368J21.3"
...,...,...,...,...,...,...
32,cGEP33,Unknown,0,0.0000,,"KLK2,KLK3,RDH11,ACPP,TMPRSS2"
26,cGEP27,Unknown,0,0.0000,,"RPL26P19,RP11-20O24.4,GAPDH,RP13-258O15.1,VIM"
30,cGEP31,Unknown,0,0.0000,,"MT-CO2,MT-CYB,MT-ATP6,MT-ND1,MT-ND4"
29,cGEP30,Unknown,0,0.0000,,"IL8,RPL13,RPL32,RP11-463O12.5,RPLP2"
